In [1]:
import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/"
response = requests.get(url)
print(response.status_code)

200


In [2]:
soup = BeautifulSoup(response.text, "lxml")

books = soup.select("article.product_pod")
print(len(books))

20


In [3]:
first = books[0]
name = first.select_one("h3 a")["title"]
print(name)

A Light in the Attic


In [4]:
price = first.select_one("p.price_color").text
stock = first.select_one("p.instock.availability").text.strip()

print(name)
print(price)
print(stock)

A Light in the Attic
Â£51.77
In stock


In [12]:
categories = soup.select("div.side_categories ul li ul li a")

rows = []

for cat in categories:
    category_name = cat.text.strip()
    category_url = url + cat["href"]

    page = requests.get(category_url)
    page_soup = BeautifulSoup(page.text, "lxml")
    category_books = page_soup.select("article.product_pod")

    for book in category_books:
        name_tag = book.select_one("h3 a")
        price_tag = book.select_one("p.price_color")
        stock_tag = book.select_one("p.instock.availability")

        if name_tag:
            name = name_tag["title"]
        else:
            name = "Unknown"

        if price_tag:
            price = price_tag.text.replace("Â", "").replace("£", "")
        else:
            price = "Unknown"

        if stock_tag:
            stock = stock_tag.text.strip()
        else:
            stock = "Unknown"

        rows.append([category_name, name, price, stock])

print(len(rows))
print(rows[0])

517
['Travel', "It's Only the Himalayas", '45.17', 'In stock']


In [13]:
with open("books.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Category", "Name", "Price", "In_Stock"])
    writer.writerows(rows)